# Modelación con tres algoritmos

En esta etapa se compararán tres modelos de clasificación utilizando las observaciones independientes preparadas durante el ETL. Todos los modelos se entrenarán y evaluarán con los mismos conjuntos para que sus resultados sean comparables.

## Librerías necesarias

Se utilizarán herramientas de Pandas y NumPy para cargar los datos, pipelines para aplicar correctamente las transformaciones y tres clasificadores de Scikit-learn.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

sns.set_theme(style="white")

## 1. Carga de los datos preparados

Se cargarán `datos_train.csv` y `datos_test.csv`. Ambos archivos contienen las 120 características estadísticas y la etiqueta `actividad` como última columna.

Para el modelado, las características se separarán en `X_train` y `X_test`, mientras que las etiquetas se guardarán en `y_train` y `y_test`.

In [2]:
# Ruta de la carpeta creada durante el ETL.
RUTA_DATOS_MODELO = Path("../datos_modelo")

# Cargamos ambos conjuntos y conservamos las actividades
# como texto para mantener los ceros iniciales.
datos_train = pd.read_csv(
    RUTA_DATOS_MODELO / "datos_train.csv",
    dtype={"actividad": str}
)

datos_test = pd.read_csv(
    RUTA_DATOS_MODELO / "datos_test.csv",
    dtype={"actividad": str}
)

# Separamos las 120 características de la etiqueta.
X_train = datos_train.drop(columns="actividad").to_numpy()
y_train = datos_train["actividad"].to_numpy()

X_test = datos_test.drop(columns="actividad").to_numpy()
y_test = datos_test["actividad"].to_numpy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Actividades en entrenamiento:", np.unique(y_train).size)
print("Actividades en prueba:", np.unique(y_test).size)

X_train: (12448, 120)
y_train: (12448,)
X_test: (3112, 120)
y_test: (3112,)
Actividades en entrenamiento: 16
Actividades en prueba: 16


### Interpretación

El conjunto de entrenamiento contiene 12,448 observaciones y el conjunto de prueba contiene 3,112. Cada observación tiene 120 características y su correspondiente etiqueta de actividad.

Los dos conjuntos conservan las 16 actividades y las características se encuentran correctamente alineadas con sus etiquetas.

## 2. Definición de los modelos

Se compararán tres algoritmos:

1. **Regresión logística:** aprende relaciones lineales entre las características y las actividades.
2. **SVM:** utiliza un kernel no lineal para buscar límites de separación más flexibles.
3. **Random Forest:** combina varios árboles de decisión para aprender relaciones más complejas.

La regresión logística y SVM incluirán un escalado dentro de su pipeline porque sus resultados dependen de la escala de las características. Random Forest no requiere esta transformación.

In [3]:
# Modelo 1: Regresión logística con estandarización.
modelo_logistico = Pipeline([
    ("escalado", StandardScaler()),
    (
        "modelo",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# Modelo 2: SVM con kernel no lineal y estandarización.
modelo_svm = Pipeline([
    ("escalado", StandardScaler()),
    (
        "modelo",
        SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=42
        )
    )
])

# Modelo 3: Random Forest.
modelo_random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# El diccionario permite recorrer los tres modelos usando
# exactamente el mismo procedimiento.
modelos = {
    "Regresión logística": modelo_logistico,
    "SVM": modelo_svm,
    "Random Forest": modelo_random_forest
}

print("Modelos preparados:")
for nombre in modelos:
    print("-", nombre)

Modelos preparados:
- Regresión logística
- SVM
- Random Forest


## 3. Entrenamiento y evaluación

Cada modelo se ajustará con `X_train` y `y_train`. Posteriormente generará una predicción directa para cada fila de `X_test`.

Para compararlos se calcularán la exactitud y el F1-score macro. La exactitud representa la proporción total de aciertos y el F1-score macro asigna la misma importancia a cada una de las 16 actividades.

In [4]:
# Guardaremos el resumen y las predicciones de cada modelo.
resultados_modelos = []
predicciones_modelos = {}

for nombre, modelo in modelos.items():

    print(f"\nEntrenando: {nombre}")

    # Ajustamos el modelo únicamente con entrenamiento.
    modelo.fit(X_train, y_train)

    # Generamos una predicción directa para cada observación.
    y_pred = modelo.predict(X_test)

    exactitud = accuracy_score(
        y_test,
        y_pred
    )

    f1_macro = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    resultados_modelos.append({
        "modelo": nombre,
        "exactitud": exactitud,
        "f1_macro": f1_macro
    })

    predicciones_modelos[nombre] = y_pred

    print(f"Exactitud: {exactitud:.4f}")
    print(f"F1-score macro: {f1_macro:.4f}")

# Tabla comparativa final.
tabla_resultados = pd.DataFrame(resultados_modelos)
display(tabla_resultados)


Entrenando: Regresión logística
Exactitud: 0.7931
F1-score macro: 0.7824

Entrenando: SVM
Exactitud: 0.9039
F1-score macro: 0.8998

Entrenando: Random Forest
Exactitud: 0.9727
F1-score macro: 0.9707


,modelo,exactitud,f1_macro
0,Regresión logística,0.793059,0.782392
1,SVM,0.903920,0.899820
2,Random Forest,0.972686,0.970660


### Interpretación de la comparación

Los tres modelos fueron evaluados directamente sobre las mismas 3,112 observaciones del conjunto de prueba. No se realizaron agrupaciones ni combinaciones posteriores de predicciones.

La regresión logística obtuvo una exactitud de **79.31%** y un F1-score macro de **0.7824**. Este fue el desempeño más bajo, lo que sugiere que una separación lineal no representa suficientemente las diferencias entre las actividades.

SVM alcanzó una exactitud de **90.39%** y un F1-score macro de **0.8998**. Su kernel no lineal permitió representar mejor las relaciones entre las características.

Random Forest obtuvo el mejor resultado, con una exactitud de **97.27%** y un F1-score macro de **0.9707**. La cercanía entre ambas métricas indica que el desempeño fue alto y relativamente consistente entre las 16 actividades.

Con estas configuraciones iniciales, Random Forest es el modelo con mejor desempeño. Los resultados corresponden a la clasificación directa de cada observación independiente preparada durante el ETL.